# Create or Update Knowledge Assistant (Product Manuals)

**Display name** is ``{table_prefix}-power-tool-manuals-assistant`` using job parameters from the bundle (``table_prefix``). **Documents path** is ``{volume}/productmanuals`` where ``volume`` is the bundle ``var.volume`` (UC path to the documents root).

If no assistant exists for that display name, this notebook creates a Knowledge Assistant, adds a **files** knowledge source, then **syncs**. If it already exists, it **syncs** only.


Use more knowledge assistant functionalities via the REST API https://docs.databricks.com/api/workspace/knowledgeassistants or SDK https://databricks-sdk-py.readthedocs.io/en/latest/workspace/knowledgeassistants/knowledge_assistants.html#databricks.sdk.service.knowledgeassistants.KnowledgeAssistantsAPI

In [ ]:
# %pip install databricks-sdk -q

from databricks.sdk import WorkspaceClient
from manage_knowledge_assistant import *

w = WorkspaceClient()

# Defaults match databricks_etl/databricks.yml variables (overridden by job base_parameters)
dbutils.widgets.text("table_prefix", "app")
dbutils.widgets.text("volume", "/Volumes/data_extraction/default/documents")
table_prefix = dbutils.widgets.get("table_prefix")
volume_base = dbutils.widgets.get("volume")

DOCUMENTS_VOLUME_PATH = f"{volume_base.rstrip('/')}/productmanuals"
DISPLAY_NAME = f"{table_prefix}-power-tool-manuals-assistant"

DESCRIPTION = (
    "Answers questions about cordless drill/driver manuals from Bosch, Makita, DeWalt, and Milwaukee. "
    "Can help with safety instructions, maintenance procedures, troubleshooting, warranty details, "
    "and technical specifications that go beyond the structured extraction."
)
INSTRUCTIONS = (
    "You are a power tool expert assistant. Answer questions using the product manuals provided. "
    "When comparing tools, reference specific model numbers and manufacturers. "
    "Always cite which manual the information comes from. "
    "If the manuals are in multiple languages, prefer the English sections."
)

KNOWLEDGE_SOURCE_DISPLAY_NAME = "power-tool-pdf-manuals"
KNOWLEDGE_SOURCE_DESCRIPTION = (
    "Original PDF manuals for cordless drill/drivers from Bosch, Makita, DeWalt, and Milwaukee."
)

print(f"DISPLAY_NAME={DISPLAY_NAME}")
print(f"DOCUMENTS_VOLUME_PATH={DOCUMENTS_VOLUME_PATH}")


In [ ]:
ka_id = get_knowledge_assistant_id_by_display_name(w, DISPLAY_NAME)

if ka_id is None:
    response = create_knowledge_assistant(w, DISPLAY_NAME, DESCRIPTION, INSTRUCTIONS)
    ka_id = response.get("id")
    print("Knowledge Assistant created.")
    print(f"id: {ka_id}")
    source_response = create_knowledge_source_files(
        w,
        ka_id,
        display_name=KNOWLEDGE_SOURCE_DISPLAY_NAME,
        description=KNOWLEDGE_SOURCE_DESCRIPTION,
        volume_path=DOCUMENTS_VOLUME_PATH,
    )
    print("Knowledge Source (files) created.")
    print(f"  source id: {source_response.get('id')}")
    status = get_knowledge_assistant(w, ka_id)
    print("Knowledge Assistant state:", status.get("state"))
    source_status = get_knowledge_source(w, ka_id, source_response.get("id"))
    print("Knowledge Source state:", source_status.get("state"))
else:
    print(f"Knowledge Assistant already exists (id={ka_id}). Syncing knowledge sources.")

sync_knowledge_sources(w, ka_id)
print("Knowledge sources sync completed.")
